In [3]:
import os


def print_dir_structure_to_file(path, output_file, ignore_list=None):
    """
    Prints the directory structure of the given path to a text file,
    ignoring specified files/folders.

    Args:
        path (str): Path to the directory to read
        output_file (str): Path to the output text file
        ignore_list (list, optional): List of filenames/directory names to ignore. Defaults to None.
    """
    if ignore_list is None:
        ignore_list = []

    def _print_dir_structure(prefix, current_path, file_writer):
        try:
            entries = os.listdir(current_path)
        except FileNotFoundError:
            file_writer.write(f"Path not found: {current_path}\n")
            return
        except PermissionError:
            file_writer.write(f"Permission denied for path: {current_path}\n")
            return

        entries = [entry for entry in entries if entry not in ignore_list]
        entries.sort(
            key=lambda x: (not os.path.isdir(os.path.join(current_path, x)), x)
        )

        for i, entry in enumerate(entries):
            entry_path = os.path.join(current_path, entry)
            is_last = i == len(entries) - 1
            current_prefix = prefix + ("└── " if is_last else "├── ")

            if os.path.isdir(entry_path):
                file_writer.write(current_prefix + f"[{entry}]\n")
                _print_dir_structure(
                    prefix + ("    " if is_last else "│   "), entry_path, file_writer
                )
            else:
                file_writer.write(current_prefix + entry + "\n")

    try:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(f"Directory structure for: {path}\n")
            _print_dir_structure("", path, f)
        print(f"Directory structure written to {output_file}")
    except Exception as e:
        print(f"Error writing directory structure: {e}")


# Example usage
path = r"/workspace\Documents\Arbeitlos\MSIT"
output_file = "dir_structure.txt"
ignore_list = [
    ".git",
    "__pycache__",
    "notebooks",
    #"data",
    ".vscode",
    "node_modules",
    ".terraform",
    "builds",
    "__init__.py",
    ".pytest_cache",
    ".venv",
    ".mypy_cache",
]

print_dir_structure_to_file(path, output_file, ignore_list)
print(f"Directory structure (excluding {ignore_list}) written to {output_file}")

Directory structure written to dir_structure.txt
Directory structure (excluding ['.git', '__pycache__', 'notebooks', '.vscode', 'node_modules', '.terraform', 'builds', '__init__.py', '.pytest_cache', '.venv', '.mypy_cache']) written to dir_structure.txt


In [5]:
import os

def print_only_folders(path, output_file, ignore_list=None):
    if ignore_list is None:
        ignore_list = []

    def _build_tree(current_path, prefix=""):
        try:
            # Get only directories that aren't in the ignore list
            entries = [
                d for d in os.listdir(current_path) 
                if os.path.isdir(os.path.join(current_path, d)) and d not in ignore_list
            ]
        except (PermissionError, FileNotFoundError):
            return []

        tree_lines = []
        for i, entry in enumerate(sorted(entries)):
            is_last = (i == len(entries) - 1)
            connector = "└── " if is_last else "├── "
            
            tree_lines.append(f"{prefix}{connector}[{entry}]")
            
            # Recurse into the folder
            new_prefix = prefix + ("    " if is_last else "│   ")
            tree_lines.extend(_build_tree(os.path.join(current_path, entry), new_prefix))
            
        return tree_lines

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"Folder Structure for: {path}\n\n")
        f.write(os.path.basename(path) + "\n")
        f.write("\n".join(_build_tree(path)))

# Configuration
target_path = r"/workspace\Desktop\Demicon\cplace\pro-code"
output_name = "folder_structure_only.txt"
# Standard folders to skip
skip = [".git", "node_modules", "__pycache__", ".venv", ".idea", ".vscode"]

print_only_folders(target_path, output_name, skip)
print(f"Done! Folder tree saved to {output_name}")

Done! Folder tree saved to folder_structure_only.txt


In [2]:
!pip install pyarrow


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import argparse
import os
import pyarrow.parquet as pq

def find_parquet_compression(file_path: str):
    """
    Reads the metadata of a Parquet file to determine the compression
    codec(s) used.

    Args:
        file_path (str): The full path to the Parquet file.
    
    Returns:
        set: A set of unique compression codecs found in the file's columns.
             Returns None if the file is invalid or not found.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at '{file_path}'")
        return None

    try:
        # Open the Parquet file and get its metadata
        parquet_file = pq.ParquetFile(file_path)
        metadata = parquet_file.metadata
        
        # A Parquet file can have different compressions for different column chunks.
        # We'll collect all unique compression types found.
        compression_types = set()

        if metadata.num_row_groups == 0:
            return {'UNCOMPRESSED'} # An empty file is effectively uncompressed

        for i in range(metadata.num_row_groups):
            row_group_meta = metadata.row_group(i)
            for j in range(row_group_meta.num_columns):
                column_meta = row_group_meta.column(j)
                compression_types.add(column_meta.compression)
        
        return compression_types

    except Exception as e:
        print(f"Error reading Parquet file: {e}")
        return None

if __name__ == "__main__":
    # --- For use in a Notebook/IPython environment ---
    
    # 1. Define the file path directly in the code
    file_to_check = r"/workspace/qgi\qgi-4.0\Afghanistan.parquet"
    
    # 2. Call the function with the path
    compressions = find_parquet_compression(file_to_check)
    
    # 3. Print the result
    if compressions:
        print(f"Successfully inspected file: {file_to_check}")
        print(f"Compression type(s) found: {', '.join(compressions)}")

Successfully inspected file: /workspace/qgi\qgi-4.0\Afghanistan.parquet
Compression type(s) found: BROTLI
